# Kata 10 — Prefix Caching

> Spec: [`specs/010-prefix-caching/spec.md`](../../specs/010-prefix-caching/spec.md)  |  Arquitectura: [`ARCHITECTURE.md`](../../ARCHITECTURE.md)

## Setup

Si `ANTHROPIC_API_KEY` no está en el entorno, se pedirá aquí (sin echo).

In [ ]:
from katas._shared.bootstrap import bootstrap

client, settings = bootstrap(budget_calls=6)
print("modelo:", settings.model)

## 1. Concepto en mis palabras

Anthropic reusa el caché KV entre llamadas cuando el **prefijo** es idéntico. Si organizo el contexto como **estático primero, dinámico al final**, el primer ~90 % entra en caché y se factura ~10 %. La regla práctica: nunca pongas la fecha o el `user_id` arriba.

## 2. Por qué importa

Mismo contenido, 10× más caro si el orden es malo. La métrica que importa: `usage.cache_read_input_tokens` debería crecer turno a turno; `cache_creation_input_tokens` aparece sólo la primera vez.

## 3. Ejemplo correcto — `cache_control` en bloque estable + sufijo dinámico

La API soporta `cache_control: {type: "ephemeral"}` sobre bloques de contenido. Marco el bloque grande estable; el suffix dinámico va sin cache_control.

In [ ]:
import datetime

BIG_STATIC = (
    "Eres un asistente de soporte para Acme Corp.\n"
    "Conoces todos los productos: Atlas, Borealis, Citadel, Dome, Echo. "
    "Cada uno tiene SLA, precio y curva de soporte distintas. "
) * 200  # ~ varios miles de tokens; el prefix caching exige longitud mínima

def call_with_cache(prompt: str):
    return client.messages.create(
        system=[
            {"type": "text", "text": BIG_STATIC, "cache_control": {"type": "ephemeral"}},
        ],
        messages=[{"role": "user", "content": prompt}],
    )

r1 = call_with_cache("¿Qué SLA tiene Borealis?")
print("turno 1:")
print("  cache_creation:", r1.usage.cache_creation_input_tokens)
print("  cache_read:    ", r1.usage.cache_read_input_tokens)
print("  input_tokens:  ", r1.usage.input_tokens)

r2 = call_with_cache("¿Y Citadel?")
print("turno 2:")
print("  cache_creation:", r2.usage.cache_creation_input_tokens)
print("  cache_read:    ", r2.usage.cache_read_input_tokens)
print("  input_tokens:  ", r2.usage.input_tokens)

En el turno 2, `cache_read_input_tokens` debería ser >0 y `cache_creation` cerca de 0: el prefijo grande se reusa.

## 4. Anti-patrón — variable dinámica al inicio del prefijo

In [ ]:
def call_breaking_cache(prompt: str):
    now = datetime.datetime.utcnow().isoformat()
    poisoned = f"NOW={now}\n\n" + BIG_STATIC  # timestamp arruina el prefijo
    return client.messages.create(
        system=[
            {"type": "text", "text": poisoned, "cache_control": {"type": "ephemeral"}},
        ],
        messages=[{"role": "user", "content": prompt}],
    )

r3 = call_breaking_cache("¿Qué SLA tiene Borealis?")
print("turno A (prefijo poisoned):")
print("  cache_read:    ", r3.usage.cache_read_input_tokens)

r4 = call_breaking_cache("¿Y Citadel?")
print("turno B (otro timestamp):")
print("  cache_read:    ", r4.usage.cache_read_input_tokens)

`cache_read` se queda en cero: cada llamada genera un prefijo distinto y nunca golpea el caché. Mismo contenido, costo 10×.

## 5. Argumento de certificación

- **Static-prefix-first, dynamic-suffix-last.** Es la única regla.
- **`cache_control: ephemeral`** en bloques largos y estables (system, CLAUDE.md, contexto pesado).
- **El timestamp/user_id va al final**, idealmente envuelto en un tag (`<reminder>`) que lo aísla.
- **Métricas auditables**: `cache_creation_input_tokens` (primera vez) vs `cache_read_input_tokens` (turnos siguientes). Es lo que el revisor mira para confirmar que el caché funciona.
- **Conexión con otros katas**: el `CLAUDE.md` mergeado del Kata 8 va aquí, en el bloque cacheable; las reglas condicionales del Kata 9 cambian el prefix sólo cuando el target del turno cambia, lo cual es un tradeoff aceptable.

## 6. Auto-evaluación

**1. ¿Dónde meto la fecha actual sin romper el caché?**

En el sufijo del último mensaje del usuario, idealmente dentro de un tag (`<reminder>now=...</reminder>`). El prefijo (system + history estable) sigue siendo idéntico turno a turno.

**2. Si el system prompt cambia un carácter, ¿qué pasa con el caché?**

Se invalida — el prefijo deja de coincidir. Por eso los `@imports` (Kata 8) ayudan: cambias un archivo importado y solo se invalida si efectivamente el contenido cambia.

**3. ¿Cómo demuestro empíricamente el ahorro?**

Comparando `cache_read_input_tokens` entre la versión correcta y la rota. La diferencia es directa: en la celda §4 cache_read=0 en ambos turnos; en §3 cache_read >> 0 en el turno 2.